## 2. Preprocesado

In [ ]:
!pip install beautifulsoup4
!pip install spacy
!pip install nltk

In [25]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from spacy.lang.en.stop_words import STOP_WORDS
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')  #  para lematización
from nltk.stem import WordNetLemmatizer
import unicodedata
import re
import string

[nltk_data] Downloading package wordnet to /Users/maru/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/maru/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


### 2.1 Cargo el dataset

In [12]:
# Cargar el dataset balanceado desde el Notebook 1
df = pd.read_pickle('Data/df_beauty_balanced.pkl')
print(f"{len(df)} reviews")
print(f"Columnas: {df.columns.tolist()}")
df.head()

6000 reviews
Columnas: ['review', 'rating']


,review,rating
0,"Nice, wrong color Loved the braid, but didn't ...",3.0
1,It does nothing. I have terrible eye bags and ...,1.0
2,These do not fit. This are hard to put on the ...,1.0
3,Fix this Only about 50% usable. Dried and sti...,2.0
4,Keep your coins Where do I start OK first off ...,1.0


### 2.2 Etiqueto las rating en positivo [1] y negativo [0]

- `0`: las menores de 3
- `1`: las mayor o igual a 3

In [17]:
def label_sentiment(row):
    if int(row['rating']) < 3:
        return 1
    else:
        return 0

In [19]:
# Ejemplo
ejemplo_row = df.iloc[0]
print(f"Original rating: {ejemplo_row['rating']}")
print(f"Sentiment label: {label_sentiment(ejemplo_row)}")

Original rating: 3.0
Sentiment label: 0


### 2.3 Convierto a minúsculas

In [13]:
def a_minusculas(texto):
    return texto.lower()

In [37]:
# Ejemplo
print(f"Original: {df.iloc[0]['review']}")
print(f"Procesado: {a_minusculas(df.iloc[0]['review'])}")

Original: Nice, wrong color Loved the braid, but didn't come in the color I ordered.
Procesado: nice, wrong color loved the braid, but didn't come in the color i ordered.


### 2.4 Aplico beatifulSoup para eliminar las etiquetas HTLM que pueda haber en el texto

In [20]:
def remove_html_tags(text):
    soup = BeautifulSoup(text, 'html.parser')
    return soup.get_text(separator=' ')

In [21]:
# Ejemplo
print(f"Original: {df.iloc[0]['review']}")
print(f"Procesado: {remove_html_tags(df.iloc[0]['review'])}")

Original: Nice, wrong color Loved the braid, but didn't come in the color I ordered.
Procesado: Nice, wrong color Loved the braid, but didn't come in the color I ordered.


### 2.5 Elimino signos de puntuación del texto

In [22]:
def eliminar_puntuacion(texto):
    translator = str.maketrans('', '', string.punctuation)
    return texto.translate(translator)

In [33]:
# Ejemplo
print(f"Original: {df.iloc[62]['review']}")
print(f"Procesado: {eliminar_puntuacion(df.iloc[62]['review'])}")

Original: Just an ordinary under arm! Natural might be, but no scent?
Procesado: Just an ordinary under arm Natural might be but no scent


### 2.6 Estandarizo caracteres especiales y elimino tildes (á→a, é→e, etc)

In [58]:
def normalizar_unicode(texto):
    return texto.encode('ascii', 'ignore').decode('ascii')

In [59]:
# Ejemplo
print(f"Original: {df.iloc[0]['review']}")
print(f"Procesado: {normalizar_unicode(df.iloc[0]['review'])}")

Original: Nice, wrong color Loved the braid, but didn't come in the color I ordered.
Procesado: Nice, wrong color Loved the braid, but didn't come in the color I ordered.


### 2.7 Eliminar números del texto 

- Los números en reviews de productos de belleza suelen ser poco informativos para análisis de sentimiento y no aportan valor semantico al modelo. 
- Reducimos ruido, simplificamos vocabulario

In [38]:
def eliminar_numeros(texto):
    return re.sub(r'\d+', '', texto)

In [42]:
# Ejemplo
print(f"Original: {df.iloc[2025]['review']}")
print(f"Procesado: {eliminar_numeros(df.iloc[2025]['review'])}")

Original: Eh Very sticky ! Wounding cure under uv light. Only lasted maybe 2/3 days. Maybe it was a user error
Procesado: Eh Very sticky ! Wounding cure under uv light. Only lasted maybe / days. Maybe it was a user error


### 2.8 Eliminar espacios blancos y espacios al inicio/final del texto.

In [61]:
def normalizar_espacios(texto):
    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip()

In [62]:
# Ejemplo
print(f"Original: {df.iloc[5520]['review']}")
print(f"Procesado: {normalizar_espacios(df.iloc[5520]['review'])}")

Original: Perfect roundout to my summer curly routine This very much helps to moisturize in a serious way.  Love this shampoo.
Procesado: Perfect roundout to my summer curly routine This very much helps to moisturize in a serious way. Love this shampoo.


### 2.9 Eliminar stopwords

Pero **manteniendo palabras de sentimiento negativo** como 'not', 'no', 'never', etc.

In [ ]:
from nltk.corpus import stopwords

# Descargar stopwords si no están disponibles
import nltk
nltk.download('stopwords', quiet=True)

def eliminar_stopwords(texto):
    # Obtener stopwords en inglés
    stop_words = set(stopwords.words('english'))
    
    # Palabras de negación a mantener
    negation_words = {'not', "n't", 'no', 'never', 'neither', 'nobody', 'nothing', 
                      'nowhere', 'none', 'nor', "don't", "doesn't", "didn't", 
                      "won't", "wouldn't", "shouldn't", "can't", "cannot", "couldn't"}
                      
    stop_words = stop_words - negation_words
    
    # Filtrar palabras
    palabras = texto.split()
    palabras_filtradas = [palabra for palabra in palabras if palabra.lower() not in stop_words]
    
    return ' '.join(palabras_filtradas)

In [44]:
# Ejemplo
print(f"Original: {df.iloc[0]['review']}")
print(f"Procesado: {eliminar_stopwords(df.iloc[0]['review'])}")

Original: Nice, wrong color Loved the braid, but didn't come in the color I ordered.
Procesado: Nice, wrong color Loved braid, didn't come color ordered.


### 2.10 Tokenización separando el texto en palabras

In [45]:
from nltk.tokenize import word_tokenize

def tokenizar(texto):
    return word_tokenize(texto)

In [46]:
# Ejemplo
print(f"Original: {df.iloc[0]['review']}")
print(f"Procesado: {tokenizar(df.iloc[0]['review'])}")

Original: Nice, wrong color Loved the braid, but didn't come in the color I ordered.
Procesado: ['Nice', ',', 'wrong', 'color', 'Loved', 'the', 'braid', ',', 'but', 'did', "n't", 'come', 'in', 'the', 'color', 'I', 'ordered', '.']


### 2.11 Lematización para usar en los modelos de ML con TF-IDF (Reduzco la dimensionalidad y ayudo al modelo a aprender mejor los datos aumentando la coincidencia de palabras)

In [47]:
from nltk.stem import WordNetLemmatizer

def lematizar_texto(tokens):
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(token) for token in tokens]

In [49]:
# Ejemplo
print(f"Original: {df.iloc[0]['review']}")
tokens = tokenizar(df.iloc[0]['review'])
print(f"Procesado: {lematizar_texto(tokens)}")

Original: Nice, wrong color Loved the braid, but didn't come in the color I ordered.
Procesado: ['Nice', ',', 'wrong', 'color', 'Loved', 'the', 'braid', ',', 'but', 'did', "n't", 'come', 'in', 'the', 'color', 'I', 'ordered', '.']


### 2.12 Eliminar tokens de menos de 3 caracteres (tokens cortos)

- Tokens de 1-2 caracteres suelen ser poco informativos (artículos, preposiciones)
- Mantiene palabras significativas como 'not', 'bad', 'buy', 'use'

In [56]:
def filtrar_tokens_cortos(tokens, min_length=3):
    return [token for token in tokens if len(token) >= min_length]

In [57]:
# Ejemplo
print(f"Original: {df.iloc[0]['review']}")
tokens = tokenizar(df.iloc[0]['review'])
print(f"Procesado: {filtrar_tokens_cortos(tokens)}")

Original: Nice, wrong color Loved the braid, but didn't come in the color I ordered.
Procesado: ['Nice', 'wrong', 'color', 'Loved', 'the', 'braid', 'but', 'did', "n't", 'come', 'the', 'color', 'ordered']


## 2.2 Pipeline completo de preprocesado

In [60]:
def preprocesado(texto, 
                   usar_stopwords=True, 
                   usar_lematizacion=True, 
                   filtrar_cortos=True,
                   min_length=3):
    
    # Verificar que la review no esté vacía
    if not texto or not isinstance(texto, str):
        return ""

    # Pasar a minusculas
    texto = a_minusculas(texto)
    
    # Eliminar etiquetas
    texto = remove_html_tags(texto)
    
    # Eliminar caracteres especiales
    texto = normalizar_unicode(texto)
    
    # Eliminar puntuación
    texto = eliminar_puntuacion(texto)
    
    # Eliminar números
    texto = eliminar_numeros(texto)
    
    # Normalizar espacios
    texto = normalizar_espacios(texto)
    
    # Eliminar stopwords (opcional)
    if usar_stopwords:
        texto = eliminar_stopwords(texto)
    
    # Tokenización
    tokens = tokenizar(texto)
    
    # Lematización (opcional)
    if usar_lematizacion:
        tokens = lematizar_texto(tokens)
    
    # Filtrar tokens cortos (opcional)
    if filtrar_cortos:
        tokens = filtrar_tokens_cortos(tokens, min_length)
    
    # Unir tokens en texto
    return ' '.join(tokens)

In [64]:
# Ejemplo completo del pipeline
review_ejemplo=df.iloc[2025]['review']

print("TEXTO ORIGINAL:")
print(review_ejemplo)

print("TEXTO PREPROCESADO:")
print(preprocesado(review_ejemplo))

print("SIN STOPWORDS NI LEMATIZACIÓN:")
print(preprocess_text(review_ejemplo, usar_stopwords=False, usar_lematizacion=False))

TEXTO ORIGINAL:
Eh Very sticky ! Wounding cure under uv light. Only lasted maybe 2/3 days. Maybe it was a user error
TEXTO PREPROCESADO:
sticky wounding cure light lasted maybe day maybe user error
SIN STOPWORDS NI LEMATIZACIÓN:
very sticky wounding cure under light only lasted maybe days maybe was user error


## 2.3 Aplicar preprocesado al dataset completo

In [74]:
# Aplicar la función de preprocesado
df['review_processed_ML'] = df['review'].progress_apply(
    lambda x: preprocesado(x, usar_stopwords=True, usar_lematizacion=True, filtrar_cortos=True)
)

Preprocesando: 100%|██████████| 6000/6000 [00:02<00:00, 2928.62it/s]


In [75]:
#Ejemplos:
print(f"ORIGINAL: {df.iloc[0]['review'][:150]}")
print(f"PROCESADO: {df.iloc[0]['review_processed_ML'][:150]}")

ORIGINAL: Nice, wrong color Loved the braid, but didn't come in the color I ordered.
PROCESADO: nice wrong color loved braid didnt come color ordered


#### Revisamos si hay alguna review vacia y eliminamos

In [76]:
empty_reviews = df[df['review_processed_ML'].str.strip() == '']
print(f"\nReviews vacías después del preprocesado: {len(empty_reviews)}")



Reviews vacías después del preprocesado: 8


In [79]:
df = df[df['review_processed_ML'].notna() & (df['review_processed_ML'].str.strip() != '')]
print(f"Total de reviews después del filtrado: {len(df)}")


Total de reviews después del filtrado: 5992


In [83]:
# Calcular reducción de vocabulario
from collections import Counter

# Vocabulario original
vocab_original = set()
for text in df['review']:
    vocab_original.update(str(text).lower().split())

# Vocabulario procesado
vocab_procesado = set()
for text in df['review_processed_ML']:
    vocab_procesado.update(str(text).split())

print(f"Vocabulario original: {len(vocab_original):,} palabras únicas")
print(f"Vocabulario procesado: {len(vocab_procesado):,} palabras únicas")
print(f"Reducción: {(1 - len(vocab_procesado)/len(vocab_original))*100:.1f}% palabras únicas")

Vocabulario original: 17,800 palabras únicas
Vocabulario procesado: 8,706 palabras únicas
Reducción: 51.1% palabras únicas


## 2.4 Guardar dataset preprocesado

Guardamos el dataset con las reviews preprocesadas para usar en el Notebook 3.

In [85]:
import os

# Crear directorio outputs si no existe
os.makedirs('outputs', exist_ok=True)

# Guardar dataset preprocesado
output_path = 'outputs/df_beauty_preprocessed_ML.pkl'
df.to_pickle(output_path)

print(f"✓ Dataset preprocesado guardado en: {output_path}")

✓ Dataset preprocesado guardado en: outputs/df_beauty_preprocessed_ML.pkl
